In [1]:
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt
from numpy.linalg import inv
import os

from colormath.color_conversions import convert_color
from colormath.color_objects import LabColor, XYZColor
from colormath.color_diff import delta_e_cie2000

import skimage
from skimage.color import rgb2lab, lab2rgb

from scipy.interpolate import BSpline, make_interp_spline

from scipy.optimize import minimize
from scipy.optimize import dual_annealing
from scipy.optimize import least_squares
from scipy.optimize import basinhopping

from numpy.random import random
from numpy.random import rand

from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

import time

In [2]:
table1 = np.load('cap1.npy')
table2 = np.load('cap3_2.npy')
table3 = np.load('cap5.npy')

In [3]:
def transfer_by_table(img, table):
    
    # reshape as nx3, and to int
    temp_img = img.reshape(-1, 3).astype(int)

    # transfer to table key
    img_keys = temp_img[:,0] + np.left_shift(temp_img[:,1], 8) + np.left_shift(temp_img[:,2], 16)

    # table mapping
    img_values = table[img_keys]

    # reconstruct to rgb image
    img_r = img_values.astype('uint8')
    img_g = np.right_shift(img_values, 8).astype('uint8')
    img_b = np.right_shift(img_values, 16).astype('uint8')

    img_trans = np.column_stack((img_r, img_g, img_b))

    img_trans = img_trans.reshape(img.shape)
    
    return img_trans

In [8]:
def Video2Video(video_file, tar_file):

    msg = ''
    
    vidcap = cv2.VideoCapture(video_file)
    
    if vidcap != None:
        
        if vidcap.isOpened():
            success = True
        else:
            msg = 'Video Open Fail'
            success = False    
    else:
        msg = 'VideoCapture None'
        success = False
        
    if success is False:
        return [False, msg, 0]
    
    # 建立資料夾
    tar_path = os.path.dirname(os.path.abspath(tar_file))
    
    if not os.path.isdir(tar_path):
        try:
            os.makedirs(tar_path)
        # 檔案已存在的例外處理
        except FileExistsError:
            print("檔案已存在。")
     
    # 開始轉換
    index = 0
    count = 0
    
    out_video = None
    
    while success:
       
        read_suc, image = vidcap.read()
        
        if read_suc is False:
            break
        
        h,w,c = image.shape
        
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        pred_img1 = transfer_by_table(image_rgb, table1)
        pred_img2 = transfer_by_table(image_rgb, table2)
        pred_img3 = transfer_by_table(image_rgb, table3)
        
        pred_img1 = cv2.cvtColor(pred_img1, cv2.COLOR_RGB2BGR)
        pred_img2 = cv2.cvtColor(pred_img2, cv2.COLOR_RGB2BGR)
        pred_img3 = cv2.cvtColor(pred_img3, cv2.COLOR_RGB2BGR)
        
        # cv2.putText(影像, 文字, 座標, 字型, 大小, 顏色, 線條寬度, 線條種類)
#        cv2.putText(pred_img1, '01', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 2, cv2.LINE_AA)
#        cv2.putText(pred_img2, '02', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 2, cv2.LINE_AA)
#        cv2.putText(pred_img3, '03', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 2, cv2.LINE_AA)

        cv2.putText(image, 'WLI', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(pred_img2, 'NBI', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 2, cv2.LINE_AA)
    
#         tar_img = np.zeros((h*2, w*2, c))
#         tar_img[:h,:w,:] = image
#         tar_img[:h,w:2*w,:] = pred_img1
#         tar_img[h:2*h,:w,:] = pred_img2
#         tar_img[h:2*h,w:2*w,:] = pred_img3
        
        tar_img = np.zeros((h, w*2, c))
        tar_img[:h,:w,:] = image
        tar_img[:h,w:2*w,:] = pred_img2
#         tar_img[h:2*h,:w,:] = pred_img2
#         tar_img[h:2*h,w:2*w,:] = pred_img3
        
        tar_img = tar_img.astype('uint8')

        if out_video is None:
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            out_video = cv2.VideoWriter(tar_file, fourcc, 30.0, (tar_img.shape[1], tar_img.shape[0]))

        out_video.write(tar_img)
        
        count += 1
        index += 1
        
#         if (index>30):
#             break
         
    if out_video is not None:
        out_video.release()
        
    vidcap.release()
    
    if success:
        msg = 'success'
    
    return [success, msg, count]

# 單一檔案轉換

In [9]:
src_file = r'C:\Users\Tony\Desktop\TEST\10_capsule\20230712\V000000091_DUMP.mp4'
tar_file = r'C:\Users\Tony\Desktop\TEST\10_capsule\20230712\V000000091_DUMP_NBI.mp4'

Video2Video(src_file, tar_file)

[True, 'success', 480]

# 批次轉換

In [5]:
import os
from os import listdir
from os.path import isfile, isdir, join
from os import walk
import pandas as pd

In [6]:
tar_path = r'F:\CCUMedicalData\capsule_dump'
tar_ext = 'mp4'

In [6]:
def filter_ext(files, tar_ext):
    
    tar_files = []
    
    for file in files:
        
        file_ext = os.path.splitext(file)[-1][1:].lower()
        
        if file_ext == tar_ext:
            tar_files.append(file)
        
    return tar_files
    
def get_all_ext_list(tar_path, tar_ext):
    
    all_tar_files = []
    
    for root, dirs, files in walk(tar_path):
        
        tar_files = filter_ext(files, tar_ext.lower())

        if len(tar_files) == 0:
            continue
            
        tar_files = [join(root, file) for file in tar_files]
            
        all_tar_files = all_tar_files + tar_files
        
    return all_tar_files

In [8]:
res_files = get_all_ext_list(tar_path, tar_ext)
res_files

['F:\\CCUMedicalData\\capsule_dump\\000_20210514_北榮IRB_20200731_P000000007_E000000007_S000000007_V000000007.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\001_20210514_北榮IRB_20200731_P000000008_E000000008_S000000008_V000000008.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\002_20210514_北榮IRB_20200731_P000000009_E000000009_S000000009_V000000009.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\003_20210514_北榮IRB_20201016_P000000010_E000000010_S000000010_V000000010.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\004_20210514_北榮IRB_20201016_P000000011_E000000011_S000000011_V000000011.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\005_20210514_北榮IRB_20201023_P000000014_E000000014_S000000014_V000000013.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\006_20210514_北榮IRB_20201023_P000000015_E000000015_S000000015_V000000014.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\007_20210514_北榮IRB_20201030_P000000016_E000000016_S000000016_V000000015.mp4',
 'F:\\CCUMedicalData\\capsule_dump\\008_20210514_北榮IRB_20201030_P000000017_E000000017_S0

In [9]:
dump_path = r'F:\CCUMedicalData\capsule_NBI_mix'

if not os.path.isdir(dump_path):
    try:
        os.makedirs(dump_path)
    # 檔案已存在的例外處理
    except FileExistsError:
        print("檔案已存在。")
        
for index, file in enumerate(res_files):
    
    dump_file = file.replace(tar_path, dump_path)
    
    Video2Video(file, dump_file)
    
    print(index, dump_file)

0 F:\CCUMedicalData\capsule_NBI_mix\000_20210514_北榮IRB_20200731_P000000007_E000000007_S000000007_V000000007.mp4
1 F:\CCUMedicalData\capsule_NBI_mix\001_20210514_北榮IRB_20200731_P000000008_E000000008_S000000008_V000000008.mp4
2 F:\CCUMedicalData\capsule_NBI_mix\002_20210514_北榮IRB_20200731_P000000009_E000000009_S000000009_V000000009.mp4
3 F:\CCUMedicalData\capsule_NBI_mix\003_20210514_北榮IRB_20201016_P000000010_E000000010_S000000010_V000000010.mp4
4 F:\CCUMedicalData\capsule_NBI_mix\004_20210514_北榮IRB_20201016_P000000011_E000000011_S000000011_V000000011.mp4
5 F:\CCUMedicalData\capsule_NBI_mix\005_20210514_北榮IRB_20201023_P000000014_E000000014_S000000014_V000000013.mp4
6 F:\CCUMedicalData\capsule_NBI_mix\006_20210514_北榮IRB_20201023_P000000015_E000000015_S000000015_V000000014.mp4
7 F:\CCUMedicalData\capsule_NBI_mix\007_20210514_北榮IRB_20201030_P000000016_E000000016_S000000016_V000000015.mp4
8 F:\CCUMedicalData\capsule_NBI_mix\008_20210514_北榮IRB_20201030_P000000017_E000000017_S000000017_V000000

78 F:\CCUMedicalData\capsule_NBI_mix\084_20220530_IRB_20210226_X030_E000000048_S000000048_V000000051.mp4
79 F:\CCUMedicalData\capsule_NBI_mix\085_20220530_IRB_20210305_X31_E000000049_S000000049_V000000052.mp4
80 F:\CCUMedicalData\capsule_NBI_mix\086_20220530_IRB_20210305_X32_E000000050_S000000050_V000000053.mp4
81 F:\CCUMedicalData\capsule_NBI_mix\087_20220530_IRB_20210312_X033_E000000051_S000000051_V000000054.mp4
82 F:\CCUMedicalData\capsule_NBI_mix\088_20220530_IRB_20210326_X034_E000000052_S000000052_V000000055.mp4
83 F:\CCUMedicalData\capsule_NBI_mix\089_20220530_IRB_20210409_X035_E000000053_S000000053_V000000056.mp4
84 F:\CCUMedicalData\capsule_NBI_mix\090_20220530_IRB_20210416_X038_E000000054_S000000054_V000000057.mp4
85 F:\CCUMedicalData\capsule_NBI_mix\091_20220530_IRB_20210423_X036_E000000056_S000000056_V000000059.mp4
86 F:\CCUMedicalData\capsule_NBI_mix\092_20220530_IRB_20210423_X037_E000000055_S000000055_V000000058.mp4
87 F:\CCUMedicalData\capsule_NBI_mix\093_20220530_IRB_202

# 批次轉換影像

In [7]:
def to_nbi(wli):
    h,w,c = wli.shape

    image_rgb = cv2.cvtColor(wli, cv2.COLOR_BGR2RGB)

    pred_img2 = transfer_by_table(image_rgb, table2)

    pred_img2 = cv2.cvtColor(pred_img2, cv2.COLOR_RGB2BGR)

    tar_img = np.zeros((h, w*2, c))
    tar_img[:h,:w,:] = wli
    tar_img[:h,w:2*w,:] = pred_img2

    tar_img = tar_img.astype('uint8')
    
    return tar_img

In [8]:
src_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule'
tar_path = r'C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi'

In [9]:
res_files = get_all_ext_list(src_path, 'jpg')

In [10]:
for index, file in enumerate(res_files):
    
    dump_file = file.replace(src_path, tar_path)
    dump_file_path = os.path.dirname(os.path.abspath(dump_file))
    
    if not os.path.isdir(dump_file_path):
        try:
            os.makedirs(dump_file_path)
        except FileExistsError:
            print("建立路徑異常。")
            
    img = cv2.imread(file)
    nbi = to_nbi(img)
    
    cv2.imwrite(dump_file, nbi)
    
    print(index, dump_file)

0 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_004246.jpg
1 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_004247.jpg
2 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_004248.jpg
3 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_004249.jpg
4 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_024338.jpg
5 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_024339.jpg
6 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_024340.jpg
7 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x001\V000000018\V000000018_024341.jpg
8 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x002\V000000019\V000000019_002613.jpg
9 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x002\V000000019\V000000019_002614.jpg
10 C:\Users\Tony\Desktop\TEST\10_capsule

85 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006843.jpg
86 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006844.jpg
87 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006845.jpg
88 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006846.jpg
89 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006847.jpg
90 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006848.jpg
91 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006849.jpg
92 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006850.jpg
93 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006851.jpg
94 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006852.jpg
95 C:\Users\Tony\Desktop\TEST\

169 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006927.jpg
170 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006928.jpg
171 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006929.jpg
172 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006930.jpg
173 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006931.jpg
174 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006932.jpg
175 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006933.jpg
176 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006934.jpg
177 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006935.jpg
178 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_006936.jpg
179 C:\Users\Tony\De

253 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007011.jpg
254 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007012.jpg
255 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007013.jpg
256 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007014.jpg
257 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007015.jpg
258 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007016.jpg
259 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007017.jpg
260 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007018.jpg
261 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007019.jpg
262 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007020.jpg
263 C:\Users\Tony\De

337 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007322.jpg
338 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007323.jpg
339 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007324.jpg
340 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007325.jpg
341 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007326.jpg
342 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007327.jpg
343 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007328.jpg
344 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007329.jpg
345 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007330.jpg
346 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x006\V000000024\V000000024_007331.jpg
347 C:\Users\Tony\De

...

927 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009822.jpg
928 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009823.jpg
929 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009824.jpg
930 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009825.jpg
931 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009826.jpg
932 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009827.jpg
933 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009828.jpg
934 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009829.jpg
935 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009830.jpg
936 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009831.jpg
937 C:\Users\Tony\De

1011 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009906.jpg
1012 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009907.jpg
1013 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009908.jpg
1014 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009909.jpg
1015 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009910.jpg
1016 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009911.jpg
1017 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009912.jpg
1018 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009913.jpg
1019 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009914.jpg
1020 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x038\V000000057\V000000057_009915.jpg
1021 C:\Us

1094 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003582.jpg
1095 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003583.jpg
1096 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003584.jpg
1097 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003585.jpg
1098 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003586.jpg
1099 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003587.jpg
1100 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003588.jpg
1101 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003589.jpg
1102 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003590.jpg
1103 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x041\V000000060\V000000060_003591.jpg
1104 C:\Us

1177 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004900.jpg
1178 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004901.jpg
1179 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004902.jpg
1180 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004903.jpg
1181 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004904.jpg
1182 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004905.jpg
1183 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004906.jpg
1184 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x042\V000000062\V000000062_004907.jpg
1185 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x044\V000000061\V000000061_004323.jpg
1186 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x044\V000000061\V000000061_004324.jpg
1187 C:\Us

1260 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005700.jpg
1261 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005701.jpg
1262 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005702.jpg
1263 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005703.jpg
1264 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005704.jpg
1265 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005705.jpg
1266 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005706.jpg
1267 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005707.jpg
1268 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005708.jpg
1269 C:\Users\Tony\Desktop\TEST\10_capsule\image_capsule_nbi\x047\V000000067\V000000067_005709.jpg
1270 C:\Us

In [8]:
import cv2
import numpy as np

In [11]:
video_file = r'D:\HS\專案\Proj-醫療-北榮計畫\研發-窄頻影像\20230220_DEMO\062_20220530_IRB_20201201_X006_E000000023_S000000023_V000000024.mp4'
vidcap = cv2.VideoCapture(video_file)


tar_file = r'C:\Users\Tony\Desktop\TEST\112.mp4'

# 開始轉換
index = 0
start_id = int(90 * 30)
end_id = int(100 * 30)

out_video = None

while True:

    read_suc, image = vidcap.read()

    if read_suc is False:
        break
        
    index = index + 1
    
    if index < start_id:
        continue
    
    if index > end_id:
        break
        
    h,w,c = image.shape
    
    img1 = image[0:880, 0:1176, :]
    img2 = image[880:1760, 0:1176, :]
    
    img1 = img1[100:-100, 100:-100,:]
    img2 = img2[100:-100, 100:-100,:]
    
    h1,w1,c1 = img1.shape

    tar_img = np.zeros((h1, w1*2, c)) 
    tar_img[:h1,:w1,:] = img1
    tar_img[:h1,w1:w1*2,:] = img2

    tar_img = tar_img.astype('uint8')

    if out_video is None:
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out_video = cv2.VideoWriter(tar_file, fourcc, 30.0, (tar_img.shape[1], tar_img.shape[0]))

    out_video.write(tar_img)

if out_video is not None:
    out_video.release()

vidcap.release()